In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [5]:
import os

print(os.listdir('/kaggle/input'))

['competitions']


In [6]:

import pandas as pd

train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

print(train.head())


   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


In [7]:

print("Training data shape:")
print(train.shape)

print("\nColumn names:")
print(train.columns.tolist())

print("\nMissing values:")
print(train.isnull().sum())


Training data shape:
(891, 12)

Column names:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

Missing values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [8]:

train_clean = train.copy()
test_clean = test.copy()

train_clean['Age'] = train_clean['Age'].fillna(train_clean['Age'].median())
test_clean['Age'] = test_clean['Age'].fillna(train_clean['Age'].median())

train_clean['Embarked'] = train_clean['Embarked'].fillna(train_clean['Embarked'].mode()[0])
test_clean['Fare'] = test_clean['Fare'].fillna(train_clean['Fare'].median())

train_clean['Sex'] = train_clean['Sex'].map({'male': 0, 'female': 1})
test_clean['Sex'] = test_clean['Sex'].map({'male': 0, 'female': 1})

print(train_clean[['Age', 'Sex', 'Embarked']].head())
print("\nMissing ages:", train_clean['Age'].isnull().sum())
print("Missing fares in test:", test_clean['Fare'].isnull().sum())


    Age  Sex Embarked
0  22.0    0        S
1  38.0    1        C
2  26.0    1        S
3  35.0    1        S
4  35.0    0        S

Missing ages: 0
Missing fares in test: 0


In [9]:

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']

X = train_clean[features].copy()
y = train_clean['Survived'].copy()
X_test = test_clean[features].copy()

X = pd.get_dummies(X, columns=['Embarked'], dtype=int)
X_test = pd.get_dummies(X_test, columns=['Embarked'], dtype=int)

X_test = X_test.reindex(columns=X.columns, fill_value=0)

print("Training features:")
print(X.head())

print("\nTarget values:")
print(y.head())

print("\nTraining shape:", X.shape)
print("Test shape:", X_test.shape)


Training features:
   Pclass  Sex   Age  SibSp  Parch     Fare  Embarked_C  Embarked_Q  \
0       3    0  22.0      1      0   7.2500           0           0   
1       1    1  38.0      1      0  71.2833           1           0   
2       3    1  26.0      0      0   7.9250           0           0   
3       1    1  35.0      1      0  53.1000           0           0   
4       3    0  35.0      0      0   8.0500           0           0   

   Embarked_S  
0           1  
1           0  
2           1  
3           1  
4           1  

Target values:
0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64

Training shape: (891, 9)
Test shape: (418, 9)


In [10]:

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_val)

accuracy = accuracy_score(y_val, predictions)

print("Validation accuracy:", round(accuracy * 100, 2), "%")


Validation accuracy: 81.56 %


In [11]:

final_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=42
)

final_model.fit(X, y)

test_predictions = final_model.predict(X_test)

print("Total predictions:", len(test_predictions))
print("First 10 predictions:", test_predictions[:10])


Total predictions: 418
First 10 predictions: [0 0 0 0 1 0 1 0 1 0]


In [12]:

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': test_predictions
})

submission.to_csv('/kaggle/working/submission.csv', index=False)

print(submission.head())
print("\nSubmission shape:", submission.shape)


   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1

Submission shape: (418, 2)
